# Cleaning the "sku_mappings"  Excel Sheet (CSV)...Again.

Why? because this time I have a better understanding of the assingment (AT LAST!).

In [4]:
from pathlib import Path
import pandas as pd

In [6]:
# Get current notebook directory
notebook_dir = Path().resolve()

# Go up one level and then into 'raws'
file_path = notebook_dir.parent / "raws" / "sku_mappings.csv"

print("Looking for:", file_path)
assert file_path.exists(), "CSV file not found at that location!"

# Read the CSV
df = pd.read_csv(file_path)
df.head(10)

Looking for: C:\Users\dell\Documents\desktop\real projects\CTSE_ASSINGMENT\raws\sku_mappings.csv


,sku,msku,panels,Status,Status.1,Unnamed: 5,image,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,15694321,15694321,Rudrav Meesho,Inactive,block,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,23654985,23654985,Rudrav Meesho,Inactive,"Combo,Paused",NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,28547595,28547595,Rudrav Meesho,Inactive,paused,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,30258741,30258741,Rudrav Meesho,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,30548764,30548764,Rudrav Meesho,Inactive,block,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,31021652,31021652,Rudrav Meesho,Inactive,paused,NaN,NaN,NaN,NaN,NaN,panels,COUNTA of panels
6,32056420,32056420,Rudrav Meesho,Inactive,block,NaN,NaN,NaN,NaN,NaN,NaN,0
7,32165478,32165478,Rudrav Meesho,Inactive,paused,NaN,NaN,NaN,NaN,NaN,CSTE AMAZON,1536
8,32565434,32565434,Rudrav Meesho,NaN,NaN,NaN,NaN,NaN,NaN,NaN,CSTE FK,801
9,32645101,32645101,Rudrav Meesho,Inactive,block,NaN,NaN,NaN,NaN,NaN,CSTE MEESHO,1360


## Step 1: Get a comprehensive view of the data

This will help us understand:
1. The exact column names (including any unnamed columns)
2. How many rows we're actually dealing with (vs summary rows)
3. The current data types
4. Any immediate issues with the structure

In [7]:
# Basic info about the dataset
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst few rows:")
print(df.head())
print("\nLast few rows:")
print(df.tail())
print("\nData types:")
print(df.dtypes)
print("\nBasic info:")
print(df.info())

Dataset shape: (5218, 12)

Column names:
['sku', 'msku', 'panels', 'Status', 'Status.1', 'Unnamed: 5', 'image', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']

First few rows:
        sku      msku         panels    Status      Status.1  Unnamed: 5  \
0  15694321  15694321  Rudrav Meesho  Inactive         block         NaN   
1  23654985  23654985  Rudrav Meesho  Inactive  Combo,Paused         NaN   
2  28547595  28547595  Rudrav Meesho  Inactive        paused         NaN   
3  30258741  30258741  Rudrav Meesho       NaN           NaN         NaN   
4  30548764  30548764  Rudrav Meesho  Inactive         block         NaN   

  image Unnamed: 7  Unnamed: 8 Unnamed: 9 Unnamed: 10 Unnamed: 11  
0   NaN        NaN         NaN        NaN         NaN         NaN  
1   NaN        NaN         NaN        NaN         NaN         NaN  
2   NaN        NaN         NaN        NaN         NaN         NaN  
3   NaN        NaN         NaN        NaN         NaN         NaN  
4 

### Now we examine the `image` columns and look for any summary rows at the bottom. 

NOTE: If you skim through the sku_mappings.csv manually and the assingment data on the [google sheets link](https://docs.google.com/spreadsheets/d/1ORu33oTA1KcLMkyjmujcBjdzfavOnkUJJJKxujFq2Fw/edit?gid=891383375#gid=891383375)

You will see that the `image` column has been spilt into two hence why we are look at two columns

This examination will help us understand:
1. The pattern in the image columns (looking for #REF!, #N/A, URLs)
2. Whether there are summary rows we need to remove
3. How to properly merge the image columns

In [ ]:
# Let's examine the image-related columns and check for patterns
print("Examining 'image' column unique values (first 20):")
print(df['image'].value_counts().head(20))
print("\nExamining 'Unnamed: 7' column unique values (first 20):")
print(df['Unnamed: 7'].value_counts().head(20))

# Let's look at some specific rows where both image columns have values
print("\nRows where both 'image' and 'Unnamed: 7' have non-null values:")
both_not_null = df[(df['image'].notna()) & (df['Unnamed: 7'].notna())]
print(f"Count: {len(both_not_null)}")
if len(both_not_null) > 0:
    print(both_not_null[['sku', 'image', 'Unnamed: 7']].head(10))

# Let's also check the last 50 rows to see if there are summary rows
print("\nLast 50 rows to check for summary data:")
print(df.tail(50)[['sku', 'msku', 'panels']])

# Check for any rows that might be summary/total rows
print("\nChecking for potential summary rows (looking for 'total', 'count', etc.):")
summary_keywords = ['total', 'count', 'sum', 'grand', 'TOTAL', 'COUNT', 'SUM', 'GRAND']
for keyword in summary_keywords:
    mask = df['sku'].astype(str).str.contains(keyword, na=False, case=False)
    if mask.any():
        print(f"Found rows with '{keyword}':")
        print(df[mask][['sku', 'msku', 'panels']])